# 02 — Frozen baseline stratified smoke training
Connect this notebook to a Colab GPU runtime and run from the top. It builds one deterministic, source- and COD10K-category-stratified 256-image manifest, trains both frozen backbones on that identical subset for five epochs, and performs the full paired diagnostic. No test set is used.

In [ ]:
# Fresh-kernel bootstrap. Edit only these settings; cached Drive assets are reused.
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
ACCEPT_COD10K_NONCOMMERCIAL_LICENSE = True

from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, torch

project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev,notebooks]'], check=True)

bootstrap_env = os.environ.copy()
dino_weights = Path('/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth')
if not dino_weights.is_file():
    print('DINOv3 requires approved Meta access on the first run only.')
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
command = [sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
           '--project-dir', str(project_dir), '--state-file', str(state_file)]
if not ACCEPT_COD10K_NONCOMMERCIAL_LICENSE:
    raise PermissionError('Review https://github.com/DengPingFan/SINet#9-license before accepting.')
command += ['--ensure-training-data', '--accept-noncommercial-license']
subprocess.run(command, cwd=project_dir, env=bootstrap_env, check=True)
bootstrap_env.pop('COD_SSL_DINOV3_DOWNLOAD_URL', None)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = state['project_dir']
DATA_ROOT = state['data_root']
RUNS_ROOT = state['runs_root']
COMPARISONS_ROOT = state['comparisons_root']
SAMPLE_IMAGE = state['sample_image']
TRAIN_MANIFEST = state['train_manifest']
print('Ready on', state['gpu'])
SMOKE_MANIFEST = str(Path(DATA_ROOT) / 'manifests/train_smoke_256_seed42.csv')
SMOKE_SELECTION_REPORT = str(Path(DATA_ROOT) / 'manifests/train_smoke_256_seed42.selection.csv')
LIMIT_TRAIN = 256
EPOCHS = 5


In [ ]:
# Validate the full training pool, then build the explicit deterministic smoke manifest.
import pandas as pd
train_all = pd.read_csv(TRAIN_MANIFEST)
counts = train_all.groupby('source').size().to_dict()
if counts != {'camo': 1000, 'cod10k': 3040}: raise ValueError(f'Unexpected training counts: {counts}')
missing_pairs = [path for column in ('image_path', 'mask_path') for path in train_all[column] if not Path(path).is_file()]
if missing_pairs: raise FileNotFoundError(f'Manifest contains missing files, first: {missing_pairs[0]}')
subprocess.run([
    sys.executable, f'{PROJECT_DIR}/scripts/prepare_smoke_manifest.py',
    '--train-manifest', TRAIN_MANIFEST, '--output', SMOKE_MANIFEST,
    '--report', SMOKE_SELECTION_REPORT, '--size', str(LIMIT_TRAIN), '--seed', '42',
], cwd=PROJECT_DIR, check=True)
smoke = pd.read_csv(SMOKE_MANIFEST)
if len(smoke) != LIMIT_TRAIN: raise RuntimeError(f'Expected {LIMIT_TRAIN} smoke rows, got {len(smoke)}')
print('Full pool:', counts, 'total=', len(train_all))
print('Smoke sources:', smoke.groupby('source').size().to_dict())
display(pd.read_csv(SMOKE_SELECTION_REPORT))
if hasattr(os, 'sync'): os.sync()

In [ ]:
# Ensure the automatically downloaded smoke-test image exists for checkpoint reload verification.
from urllib.request import urlretrieve
if not Path(SAMPLE_IMAGE).is_file():
    urlretrieve('https://raw.githubusercontent.com/DengPingFan/SINet/master/Images/CamouflagedTask.png',SAMPLE_IMAGE)
print('Sample image:',SAMPLE_IMAGE)

In [ ]:
# Run frozen DINOv3 + common decoder with the configured intermediate smoke settings.
dino_before=set(Path(RUNS_ROOT).glob('*_dinov3_vitb16_seed42'))
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/train.py','--config',f'{PROJECT_DIR}/configs/frozen_dinov3_vitb16.yaml','--runs-root',RUNS_ROOT,'--train-manifest',SMOKE_MANIFEST,'--epochs',str(EPOCHS)],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'DINOv3 smoke training failed: {result.returncode}')
dino_new=set(Path(RUNS_ROOT).glob('*_dinov3_vitb16_seed42'))-dino_before
if len(dino_new)!=1: raise RuntimeError(f'Could not identify DINOv3 run: {dino_new}')
DINO_RUN=str(dino_new.pop()); print('DINO_RUN=',DINO_RUN)

In [ ]:
# Reload the DINOv3 checkpoint and verify finite 384×384 logits and the freeze invariant.
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/verify_checkpoint.py','--run',DINO_RUN,'--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError('DINOv3 checkpoint reload failed.')

In [ ]:
# Release process-local allocator caches before V-JEPA.
import gc
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Run frozen V-JEPA 2.1 + the identical common decoder and smoke settings.
vjepa_before=set(Path(RUNS_ROOT).glob('*_vjepa21_vitb16_seed42'))
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/train.py','--config',f'{PROJECT_DIR}/configs/frozen_vjepa21_vitb16.yaml','--runs-root',RUNS_ROOT,'--train-manifest',SMOKE_MANIFEST,'--epochs',str(EPOCHS)],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'V-JEPA smoke training failed: {result.returncode}')
vjepa_new=set(Path(RUNS_ROOT).glob('*_vjepa21_vitb16_seed42'))-vjepa_before
if len(vjepa_new)!=1: raise RuntimeError(f'Could not identify V-JEPA run: {vjepa_new}')
VJEPA_RUN=str(vjepa_new.pop()); print('VJEPA_RUN=',VJEPA_RUN)

In [ ]:
# Reload the V-JEPA checkpoint and verify its output and freeze invariant.
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/verify_checkpoint.py','--run',VJEPA_RUN,'--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError('V-JEPA checkpoint reload failed.')

In [ ]:
# Summarize smoke losses and required artifacts.
import numpy as np
for name,run in [('DINOv3',DINO_RUN),('V-JEPA 2.1',VJEPA_RUN)]:
    log=pd.read_csv(Path(run)/'training_log.csv')
    if len(log)!=EPOCHS: raise RuntimeError(f'{name} logged {len(log)} epochs; expected {EPOCHS}')
    if not np.isfinite(log.loss).all(): raise RuntimeError(f'{name} produced non-finite loss')
    if log.loss.iloc[-1]>=log.loss.iloc[0]: raise RuntimeError(f'{name} loss did not decrease across the smoke run')
    required=[Path(run)/'checkpoints/last.pt',Path(run)/'samples/training_sample.png',Path(run)/'config.yaml',Path(run)/'environment.txt',Path(run)/'upstream_versions.json',Path(run)/'train_manifest.csv',Path(run)/'train_manifest.sha256']
    missing=[str(path) for path in required if not path.is_file()]
    if missing: raise FileNotFoundError(f'{name} missing artifacts: {missing}')
    print(f'\n{name}: {run}'); print(log[['epoch','loss','learning_rate','wall_time_seconds']].to_string(index=False))
    print('Prediction:',Path(run)/'samples/training_sample.png')

In [ ]:
# Export paired qualitative panels before authorizing the full Phase-1 runs.
SMOKE_VIS_DIR = Path(RUNS_ROOT) / 'smoke_visual_comparison'
visualization_script = Path(PROJECT_DIR) / 'scripts/visualize_smoke_comparison.py'
if not visualization_script.is_file():
    raise FileNotFoundError(
        f'{visualization_script} is missing from the Colab checkout. Commit/push the latest '
        'local changes, then rerun the notebook repository-sync cell before this cell.'
    )
result = subprocess.run([
    sys.executable, str(visualization_script),
    '--manifest', SMOKE_MANIFEST, '--dino-run', DINO_RUN, '--vjepa-run', VJEPA_RUN,
    '--output', str(SMOKE_VIS_DIR), '--count', '6', '--training-subset', str(LIMIT_TRAIN),
], cwd=PROJECT_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode:
    raise RuntimeError(f'Smoke visualization failed with exit code {result.returncode}.')
if hasattr(os, 'sync'): os.sync()
print('Smoke visual comparison:', SMOKE_VIS_DIR)

In [ ]:
# Display the complete 256-image diagnostic tables/plots and six score-selected panels.
from IPython.display import display
from PIL import Image
print('TRAINING-SUBSET DIAGNOSTIC — not held-out publication evidence')
print('Descriptive summary (all 256 images)')
display(pd.read_csv(SMOKE_VIS_DIR / 'model_summary.csv'))
print('Paired comparison: differences are DINOv3 minus V-JEPA')
display(pd.read_csv(SMOKE_VIS_DIR / 'paired_comparison.csv'))
print('Protocol COD metrics on the same training subset')
display(pd.read_csv(SMOKE_VIS_DIR / 'aggregate_cod_metrics.csv'))
display(Image.open(SMOKE_VIS_DIR / 'smoke_diagnostic_plots.png'))
selection = pd.read_csv(SMOKE_VIS_DIR / 'qualitative_selection.csv')
display(selection[['id', 'selection_reason', 'dino_dice', 'vjepa_dice', 'dice_difference']])
for panel in sorted((SMOKE_VIS_DIR / 'qualitative_panels').glob('*.png')):
    print(panel.name)
    display(Image.open(panel))

Milestone H passes when both models train on the identical stratified manifest, have finite decreasing losses, saved predictions, reloadable checkpoints, `[1,1,384,384]` finite logits, and zero trainable/gradient-bearing backbone parameters. Full experiments remain blocked until the revised paired diagnostics and overlays are inspected.